# Week 1 Lab: Exploring Tokenization and Embeddings

**2026–2027 · updated 7 October 2026**

Start by splitting text, then count words, compare subword tokenizers, and inspect dense embeddings. These experiments show what a model receives and how it represents text. Token counts affect the space available for data and instructions; vector similarities affect search and classification in later labs.

1. Whitespace tokens and a vocabulary.
2. Bag-of-words vectors.
3. Subword tokenizers: token IDs, coloured tokens, and count distributions.
4. Dense embeddings and neighbours; distinguish lookup vectors from contextual representations.
5. Optional extensions: the lecture's attention calculation and one hosted model call.

By the end, you should be able to explain **text → token IDs → lookup vectors → contextual vectors** and compare two texts without treating a similarity score as proof that their meanings are identical.

Work through Sections 1–4 first, using the robot/drone sentences, news sample, and neighbour exercise. BERT, GPT-2, and MiniLM remain small teaching examples; the Qwen3.5 tokenizer adds a current comparison without downloading its model weights. The larger embedding model and hosted call are optional.

## 0. Setup

From the course repository, use the shared environment and dependencies:

```bash
python3 -m venv .venv
source .venv/bin/activate  # Windows PowerShell: .\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m ipykernel install --user --name llms-data-science --display-name "LLMs Data Science"
jupyter lab labs
```

Select that kernel in Jupyter or VS Code. If your existing `venvLLMDS` environment already works, keep it and install the shared requirements there.

The tokenizer, AG News, and MiniLM sections need public downloads on their first run. They do **not** need an API key. Set `LOCAL_FILES_ONLY = True` to reuse downloaded tokenizers/models without contacting the Hub. For an offline AG News session, cache it first and launch Jupyter with `HF_DATASETS_OFFLINE=1` set in the environment. Sections 1–2, the small vector example, and the attention extension use local calculations only.

Run this notebook from the repository root or its `labs/` directory. Any private notes and exported tables go under `local/student/2026-2027/week_1/`, which Git ignores. Shared notebook outputs are cleared before committing.

### Hosted generation is optional

The last API cell is disabled with `RUN_HOSTED_GENERATION = False`. Leave it disabled for the core lab.

To try it later, choose a chat model currently offered by [Hugging Face Inference Providers](https://huggingface.co/docs/inference-providers/tasks/chat-completion). Add `HF_TOKEN`, `HF_MODEL`, and optionally `HF_PROVIDER` to your existing, ignored `.env` file. Create a fine-grained token with **Make calls to Inference Providers** permission. A public model repository does not imply that an inference provider serves that model.

Check the selected provider's access and price before enabling the call. We do not assume a free tier or a permanently available model. The cell loads `.env` only when enabled; it sends the teaching prompt, not your datasets or notebook.

In [ ]:
from __future__ import annotations

from collections import Counter
from html import escape
from importlib.metadata import version
from pathlib import Path
from typing import Iterable

from IPython.display import HTML, display
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer

LOCAL_FILES_ONLY = False  # True: tokenizers/models must already be cached.
RUN_LARGER_EMBEDDING_MODEL = False
RUN_HOSTED_GENERATION = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
COURSE_ROOT = Path.cwd().resolve()
if not (COURSE_ROOT / "labs").is_dir():
    COURSE_ROOT = COURSE_ROOT.parent
if not (COURSE_ROOT / "labs" / "week_1_tokenization.ipynb").is_file():
    raise FileNotFoundError("Start Jupyter from the course root or labs directory.")
WORK_DIR = COURSE_ROOT / "local" / "student" / "2026-2027" / "week_1"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
display(pd.DataFrame({
    "package": ["numpy", "pandas", "torch", "transformers", "datasets", "huggingface-hub"],
    "version": [version(p) for p in ["numpy", "pandas", "torch", "transformers", "datasets", "huggingface-hub"]],
}))

## 1. Whitespace tokenization → vocabulary

The lecture introduced tokenization by simply splitting text on whitespace. Recreate that pipeline for two sentences and inspect the intermediate artifacts.

Use two sentences: one about a robot, one about a drone. Lowercasing and `split()` give a simple tokenizer. Notice that punctuation remains attached: `smart` and `smart.` are different tokens. The position column starts at zero, following Python indexing.

In [ ]:
sentences = {
    "robotics": "This is a smart robot exploring language.",
    "drones": "My agile drone is also very smart."
}

whitespace_tokens = {
    name: sentence.lower().split()
    for name, sentence in sentences.items()
}

token_table = pd.DataFrame(
    [(name, i, token) for name, tokens in whitespace_tokens.items() for i, token in enumerate(tokens)],
    columns=["sentence", "position", "token"]
)

token_table

Arrange the two token lists side by side to see shared words and punctuation differences. Empty cells mean that one sentence ended earlier.

In [ ]:
# Quick side-by-side look at whitespace tokenization
comparison = (pd.DataFrame.from_dict(whitespace_tokens, orient='index')
              .T.reset_index(drop=False)
              .rename(columns={'index': 'position'}))
comparison.fillna('', inplace=True)
comparison

In [ ]:
# Build a vocabulary and basic statistics
vocabulary = sorted({token for tokens in whitespace_tokens.values() for token in tokens})
vocab_counts = Counter(token for tokens in whitespace_tokens.values() for token in tokens)

print(f"Vocabulary size: {len(vocabulary)}")
pd.DataFrame({"token": vocabulary, "frequency": [vocab_counts[tok] for tok in vocabulary]})

## 2. Manual bag-of-words vectors

Use the shared vocabulary as the coordinates of a vector. Each coordinate counts one vocabulary item in the document. Counts may exceed one; they are not restricted to 0 or 1. Word order disappears.

In [ ]:
def bag_of_words(tokens: Iterable[str], vocab: list[str]) -> np.ndarray:
    counts = Counter(tokens)
    return np.array([counts.get(term, 0) for term in vocab], dtype=np.int32)

bow_vectors = {
    name: bag_of_words(tokens, vocabulary)
    for name, tokens in whitespace_tokens.items()
}

bow_df = pd.DataFrame(bow_vectors, index=vocabulary)
bow_df

### Exercise

- Add a third sentence. Rerun the vocabulary and bag-of-words cells; which coordinates change?
- Compare `a robot follows a drone` with `a drone follows a robot`. Why are their bag-of-words vectors identical even though the roles are reversed?
- Compute cosine similarity between the original robot/drone vectors: $\cos(u,v)=\frac{u\cdot v}{\|u\|_2\|v\|_2}$. Here $\|u\|_2$ is the vector's Euclidean length. What does this score measure when the coordinates are word counts?

## 3. Subword tokenizers on a dataset

A fixed word vocabulary cannot list every name, spelling, or word form. Subword tokenizers represent text with smaller pieces drawn from a finite vocabulary. A **token ID** is the integer used to look up a piece; the same integer can refer to different pieces in different tokenizers.

First inspect BERT and GPT-2. BERT's cased and uncased variants show why changing case can change the input. The news experiment compares GPT-2, BERT, and RoBERTa with [Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B), a 2026 model. We download only its tokenizer files.

We use **BPE** (byte-pair encoding) and **WordPiece** as two examples of subword vocabularies; the [tokenization guide](https://huggingface.co/docs/transformers/main/en/tokenizer_summary) explains how they are built. A tokenization comparison is not a model-quality ranking.

In [ ]:
TOKENIZER_NAMES = ["bert-base-uncased", "gpt2", "distilroberta-base", "Qwen/Qwen3.5-0.8B"]
reference_names = ["bert-base-uncased", "bert-base-cased", "gpt2"]
# Load once and reuse in the coloured examples and count experiment.
tokenizers = {
    name: AutoTokenizer.from_pretrained(name, use_fast=True,
                                       local_files_only=LOCAL_FILES_ONLY, token=False)
    for name in dict.fromkeys(reference_names + TOKENIZER_NAMES)
}

for model_name in reference_names:
    tokenizer = tokenizers[model_name]
    print(f"\n=== Tokenization with {model_name} ===")
    for label, sentence in sentences.items():
        # These IDs and pieces correspond exactly, including any special tokens.
        ids = tokenizer.encode(sentence, add_special_tokens=True)
        display(pd.DataFrame({
            "position": range(len(ids)),
            "token": tokenizer.convert_ids_to_tokens(ids),
            "token_id": ids,
            "special": [token_id in tokenizer.all_special_ids for token_id in ids],
        }))
        print(label, ":", sentence)

In [ ]:
# Load the AG News sample once; reuse it for examples and count comparisons.
ds = load_dataset("ag_news", split="train[:1000]", token=False)
news_texts = list(ds["text"])
examples = {
    "short": news_texts[5],
    "medium": " ".join(news_texts[20:22]),
    "long": " ".join(news_texts[50:60]),
    "French": "Le petit robot rouge analyse les données à Marseille.",
    "Python": 'return df.groupby("group")["value"].mean()',
    "symbols": "robot <tag> & data: café 🤖 12.5 mg/L",
}

for name in TOKENIZER_NAMES:
    print(f"{name} → vocabulary size, including added tokens: {len(tokenizers[name])}")


def prettify_tokens(tokens):
    # Display common space markers; keep WordPiece's ## continuation marker.
    return [t.replace("Ġ", "␠").replace("▁", "␠") for t in tokens]


def colorize_tokens(tokens):
    colors = ["#eef6ff", "#FFD4CC", "#FABF8F", "#FFFE85", "#DCCBFE"]
    return " ".join(
        f'<span style="background:{colors[i % len(colors)]}; padding:2px; margin:1px; '
        f'border-radius:3px; font-family:monospace;">{escape(t)}</span>'
        for i, t in enumerate(tokens)
    )


def show_tokenization(model_name, text):
    tok = tokenizers[model_name]
    ids = tok.encode(text, add_special_tokens=False)
    pieces = tok.convert_ids_to_tokens(ids)
    view = (
        f"<h4>{escape(model_name)}</h4>"
        f"<div>{colorize_tokens(prettify_tokens(pieces))}</div>"
        f"<div style='margin-top:6px; font-family:monospace;'>text tokens = {len(ids)}</div>"
    )
    return view, pieces, ids


for label, text in examples.items():
    display(HTML(f"<h3>Example: {escape(label)}</h3><p>{escape(text)}</p>"))
    for model_name in TOKENIZER_NAMES:
        view, pieces, ids = show_tokenization(model_name, text)
        display(HTML(view))

### Compare counts on the same articles

AG News is a historical news dataset; here we study its tokenization, not current events. Count the **same 1,000 texts** with each tokenizer, without truncation or added special tokens. This isolates text tokenization.

In the coloured view, `␠` displays a space marker and BERT's `##` marks a continuation. The colours separate tokens; they do not show attention or importance. Byte-level pieces can look unusual for accents and emoji. Inspect the original text alongside the pieces.

In [ ]:
records = []
for model_name in TOKENIZER_NAMES:
    lengths = [len(ids) for ids in tokenizers[model_name](
        news_texts, add_special_tokens=False, truncation=False
    )["input_ids"]]
    for article, count in enumerate(lengths):
        records.append({"article": article, "model": model_name, "tokens": count})

token_counts = pd.DataFrame(records)
token_stats = token_counts.groupby("model")["tokens"].agg(
    articles="count", mean="mean", median="median", maximum="max"
)
display(token_stats.round(2))

fig = px.histogram(
    token_counts, x="tokens", color="model", barmode="overlay", nbins=40,
    title="Text token counts on the same AG News articles (no added special tokens)",
)
fig.update_layout(bargap=0.05)
fig.show()

## 4. Dense embeddings and neighbours

A **lookup embedding** is the learned vector selected by a token ID. It starts the model's computation. Transformer blocks then produce **contextual token vectors**: vectors that can depend on other positions. To represent a whole sentence for similarity, this model averages its final contextual token vectors; that averaging is called **mean pooling**.

[all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) is a compact sentence-similarity model that runs on a laptop. The dot product of two such vectors is their cosine similarity.

A **padding token** fills an empty batch position so the inputs have equal lengths. The `attention_mask` marks real input positions with 1 and padding with 0. Here we also use it to exclude padding from the average. This is different from the decoder's mask that blocks future tokens.

### First check the geometry without a model download

The following invented 2-D vectors explain cosine similarity. Their coordinates are chosen for the exercise, not learned meanings. Compare a vector with a scaled copy and with a perpendicular vector. Cosine uses direction, not length.

In [ ]:
def cosine_matrix(vectors: np.ndarray) -> np.ndarray:
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(lengths == 0):
        raise ValueError("Cosine similarity is undefined for a zero vector.")
    unit_vectors = vectors / lengths
    return unit_vectors @ unit_vectors.T


toy_vectors = np.array([[1., 0.], [2., 0.], [0.9, 0.1], [0., 1.]])
toy_labels = ["first", "scaled copy", "nearby", "perpendicular"]
display(pd.DataFrame(cosine_matrix(toy_vectors), index=toy_labels, columns=toy_labels))

In [ ]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_tokenizer = AutoTokenizer.from_pretrained(
    embedding_model_name, local_files_only=LOCAL_FILES_ONLY, token=False
)
embedding_model = AutoModel.from_pretrained(
    embedding_model_name, local_files_only=LOCAL_FILES_ONLY, token=False
).to(DEVICE)
embedding_model.eval()  # Disable training-time dropout during this comparison.


def encode_texts(texts: list[str]) -> torch.Tensor:
    encoded = embedding_tokenizer(
        texts, padding=True, truncation=True, max_length=256, return_tensors="pt"
    ).to(DEVICE)
    with torch.inference_mode():
        model_output = embedding_model(**encoded)
        # Pool final contextual vectors, excluding padding; then normalise.
        mask = encoded["attention_mask"].unsqueeze(-1).to(model_output.last_hidden_state.dtype)
        pooled = (model_output.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        embeddings = torch.nn.functional.normalize(pooled, p=2, dim=1)
    return embeddings.cpu()


words = ["cats", "dog", "puppy", "houses", "apple", "robot", "drone"]
embeddings = encode_texts(words)
cos = embeddings @ embeddings.T
similarity_df = pd.DataFrame(cos.numpy(), index=words, columns=words)
display(similarity_df.round(3))
print("Embedding matrix shape:", tuple(embeddings.shape))
px.imshow(similarity_df, text_auto=".2f", zmin=-1, zmax=1,
          color_continuous_scale="RdBu_r", title="MiniLM cosine similarities").show()

### Discussion and exercise

- Which pairs are nearest? Each row has a similarity with itself; exclude that entry when finding a neighbour.
- Replace the word list with sentences or domain-specific terms. Try negation: does a similar vector guarantee the same claim?
- Explain why `encode_texts` returns one vector per input text. When the text contains one word, the result is still a **pooled sentence-model output**, not the raw vocabulary lookup vector.
- Text beyond 256 tokens is truncated in this exercise. What could disappear if a key qualification occurs at the end?

### The same token, two contexts

Compare `robot` in **A red toy robot moves.** and **A red industrial robot moves.** Its token ID and lookup vector are the same. The final contextual vector comes from running the whole encoder, including its attention and feed-forward layers, on each sentence. The table shows the first six coordinates, not semantic labels.

`get_input_embeddings()` gives the vocabulary lookup table; `last_hidden_state` gives the final vector at every input position. We isolate the `robot` position rather than pooling the sentence. This is an observed model calculation; the small attention exercise below uses invented matrices so we can inspect its arithmetic.

In [ ]:
contexts = ["A red toy robot moves.", "A red industrial robot moves."]
context_batch = embedding_tokenizer(contexts, padding=True, return_tensors="pt").to(DEVICE)
robot_ids = embedding_tokenizer.encode("robot", add_special_tokens=False)
assert len(robot_ids) == 1, "This illustration assumes MiniLM's robot is one token."
positions = [
    (row == robot_ids[0]).nonzero(as_tuple=True)[0].item()
    for row in context_batch["input_ids"]
]
with torch.inference_mode():
    lookup_vectors = embedding_model.get_input_embeddings()(context_batch["input_ids"])
    contextual_vectors = embedding_model(**context_batch).last_hidden_state
    robot_lookup = torch.stack([lookup_vectors[i, pos] for i, pos in enumerate(positions)]).cpu()
    robot_contextual = torch.stack([contextual_vectors[i, pos] for i, pos in enumerate(positions)]).cpu()

coordinate_table = pd.DataFrame(
    torch.cat([robot_lookup, robot_contextual]).numpy()[:, :6],
    index=["lookup: red toy", "lookup: red industrial", "contextual: red toy", "contextual: red industrial"],
    columns=[f"coordinate {i + 1}" for i in range(6)],
)
display(coordinate_table.round(4))
print("Lookup vectors identical:", torch.equal(robot_lookup[0], robot_lookup[1]))
print("Contextual vectors identical:", torch.equal(robot_contextual[0], robot_contextual[1]))

### Optional: compare a newer embedding model

[Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) is a newer multilingual embedding example. It is much larger than MiniLM. Leave this cell disabled unless you have time and enough memory for its download and forward pass.

Use `SentenceTransformer` here because it loads this model's pooling configuration. Its recipe differs from MiniLM's mean pooling; changing a model ID in our handcrafted `encode_texts` function is not a sufficient model migration. For retrieval, follow the model card's query-instruction format; this experiment compares the same short texts symmetrically. Compare neighbour rankings, dimensions, and elapsed time rather than assuming that a newer model improves every task.

In [ ]:
if RUN_LARGER_EMBEDDING_MODEL:
    from time import perf_counter
    from sentence_transformers import SentenceTransformer

    newer_name = "Qwen/Qwen3-Embedding-0.6B"
    newer_model = SentenceTransformer(
        newer_name, device=str(DEVICE), local_files_only=LOCAL_FILES_ONLY, token=False
    )
    newer_model.max_seq_length = 256  # Keep the short-text experiment bounded.
    started = perf_counter()
    newer_vectors = newer_model.encode(words, normalize_embeddings=True, show_progress_bar=False)
    newer_cos = pd.DataFrame(newer_vectors @ newer_vectors.T, index=words, columns=words)
    print(newer_name, "shape:", newer_vectors.shape,
          "encoding seconds:", round(perf_counter() - started, 3))
    display(newer_cos.round(3))
else:
    print("Larger embedding comparison skipped; MiniLM is the core exercise.")

### 4b. Optional: follow the lecture's self-attention calculation

Follow the lecture's self-attention calculation with NumPy, using **red toy robot** throughout.

An attention calculation combines information from token positions. In **self-attention**, queries, keys, and values come from the same incoming sequence. Each row of `X` is a token's current vector. `X @ WQ`, `X @ WK`, and `X @ WV` multiply it by three learned maps: a **query** for comparing positions, a **key** to compare against, and a **value** carrying the information to combine. The maps stay fixed during inference.

These matrices are invented teaching parameters. They do not come from Qwen or MiniLM, and their coordinates are not literal meanings. Identity input and value matrices keep the arithmetic simple; position information and normalisation are omitted.

In [ ]:
tokens = ["red", "toy", "robot"]
X = np.eye(3)
WQ = np.array([[1., 0.], [0., 1.], [1., 1.]])
WK = np.array([[2., 0.], [0., 2.3], [0., 0.2]])
WV = np.eye(3)
Q, K, V = X @ WQ, X @ WK, X @ WV
for name, matrix in [("X: current vectors", X), ("Q: queries", Q), ("K: keys", K), ("V: values", V)]:
    print(name)
    display(pd.DataFrame(matrix, index=tokens).round(3))

#### Compare positions, then calculate weights

The **dot product** multiplies corresponding coordinates and sums the products. `Q @ K.T` computes every query–key dot product; `.T` transposes key rows into columns. Rows receive information and columns supply it. Divide by the square root of the query/key dimension (2 here) to obtain scaled scores.

For decoder self-attention, block future positions before calculating weights: replace their scores with $-\infty$. **Softmax** exponentiates the remaining scores and divides by the row sum. The weights sum to one in each row; they are not percentages of a token's meaning or next-token probabilities.

In [ ]:
raw_scores = Q @ K.T
scaled_scores = raw_scores / np.sqrt(Q.shape[1])
future_positions = np.triu(np.ones((len(tokens), len(tokens)), dtype=bool), k=1)
masked_scores = np.where(future_positions, -np.inf, scaled_scores)

# Subtract the maximum for numerical stability; softmax is unchanged.
row_max = masked_scores.max(axis=1, keepdims=True)
exp_scores = np.exp(masked_scores - row_max)
A = exp_scores / exp_scores.sum(axis=1, keepdims=True)

print("robot query · toy key:", Q[2] @ K[1])
print("Scaled scores (receiving rows, supplying columns):")
display(pd.DataFrame(scaled_scores, index=tokens, columns=tokens).round(3))
attention_df = pd.DataFrame(A, index=tokens, columns=tokens)
px.imshow(attention_df, text_auto=".3f", zmin=0, zmax=1,
          labels={"x": "Supplies a value", "y": "Receives information", "color": "Weight"},
          title="Decoder self-attention: future positions have zero weight").show()

#### Combine values, then add to the original vector

`H = A @ V` forms a weighted combination of value vectors for every position. It is the **attention contribution**. A **residual connection** carries the original vector around attention so we can add the contribution to it. We use an identity output projection here, so the addition is simply `X + H`.

A weight of about 0.111 on `robot`'s own value does **not** mean that it retains only 11.1% of its original representation. Inspect the separate addition. Feed-forward processing and normalisation still complete the block; the **next block** then computes fresh queries, keys, and values from its incoming vectors using its own maps.

In [ ]:
H = A @ V
Y = X + H
robot_update = pd.DataFrame(
    [X[2], H[2], Y[2]],
    index=["Original robot vector", "Attention contribution", "After addition"],
    columns=["coordinate 1", "coordinate 2", "coordinate 3"],
)
display(robot_update.round(3))
print("Input token labels:", tokens, "— unchanged.")
print("Every row's attention weights sum to:", A.sum(axis=1))

**Try it:** increase the second coordinate of `toy`'s key in `WK`, then rerun the three calculation cells. Predict what will happen to `robot`'s attention weights and added vector before looking. Why do the other weights in that row change too? Remove the future mask and explain which earlier positions can now use information from `robot`.

This is one attention calculation inside one model pass. The feedback loops at the end of the course instead connect model outputs to tool execution and observations; they are a different kind of repetition.

## 5. Optional: Hosted generation via Hugging Face Inference Providers

A local embedding forward pass returns vectors. A **chat completion** returns generated text. Use a short tokenization prompt with the [chat-completion interface](https://huggingface.co/docs/inference-providers/tasks/chat-completion) and a model served by your chosen provider.

Set `HF_MODEL` to a chat model supported by your selected provider. Enable `RUN_HOSTED_GENERATION` only when you want one live call. The cell records model/provider settings, elapsed time, and token usage when supplied. Check the answer against your tokenization experiment. This single call has no tool execution or feedback loop; we build those in later labs.

In [ ]:
if RUN_HOSTED_GENERATION:
    import os
    from time import perf_counter
    from dotenv import load_dotenv
    from huggingface_hub import InferenceClient

    load_dotenv(COURSE_ROOT / ".env", override=False)
    hf_token = os.getenv("HF_TOKEN")
    hf_model = os.getenv("HF_MODEL")
    hf_provider = os.getenv("HF_PROVIDER", "auto")
    if not hf_token or not hf_model:
        raise ValueError("Set HF_TOKEN and HF_MODEL in the local environment before enabling the call.")

    prompt = "Summarise why subword tokenization is helpful for transformer models."
    client = InferenceClient(provider=hf_provider, api_key=hf_token, timeout=30)
    started = perf_counter()
    response = client.chat.completions.create(
        model=hf_model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=160,
    )
    print(response.choices[0].message.content)
    display(pd.DataFrame([{
        "requested_model": hf_model,
        "reported_model": getattr(response, "model", None),
        "provider_setting": hf_provider,
        "elapsed_s": round(perf_counter() - started, 3),
        "prompt_tokens": getattr(response.usage, "prompt_tokens", None),
        "completion_tokens": getattr(response.usage, "completion_tokens", None),
    }]))
else:
    print("Hosted generation skipped. The core lab needs no API key.")

## 6. Reflection

- Compare whitespace and subword token counts. Explain punctuation, case, and at least one French, code, or symbol example.
- Explain why fewer tokens alone do not establish a better model.
- Compare two embedding neighbours with your expectations. Give one case where high similarity could mislead a downstream task.
- Explain how the same `robot` lookup vector leads to different contextual vectors. If you ran the attention extension, distinguish `H` from `X + H` and explain when new queries and keys are calculated.
- Give a task where a missing definition or unit would change the answer. State the context you would supply and one independent check for a plausible but wrong result.
- Be able to explain the functions and terms in your code. You may use AI to help, but verify its suggestions with the experiments and describe what you checked.

Next week we use the same tokenizer/model separation when loading pretrained models and tracing their forward pass and generation.
